# 01 - Check Project and Dataset

Use this notebook first. It verifies the dataset folders, class order, label files, and a few sample annotations before training.

In [ ]:
from pathlib import Path
import random

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_DIR = PROJECT_ROOT / 'datasets' / 'fire_smoke_roboflow_v4'
DATA_YAML = DATASET_DIR / 'data_project.yaml'

print('Project root:', PROJECT_ROOT)
print('Dataset dir:', DATASET_DIR)
print('Dataset yaml:', DATA_YAML)
print('Dataset exists:', DATASET_DIR.exists())
print('YAML exists:', DATA_YAML.exists())

: 

In [ ]:
splits = ['train', 'valid', 'test']
for split in splits:
    image_dir = DATASET_DIR / split / 'images'
    label_dir = DATASET_DIR / split / 'labels'
    images = list(image_dir.glob('*')) if image_dir.exists() else []
    labels = list(label_dir.glob('*.txt')) if label_dir.exists() else []
    print(f'{split:5s} images: {len(images):5d} | labels: {len(labels):5d}')

In [ ]:
print(DATA_YAML.read_text())

In [ ]:
label_files = list((DATASET_DIR / 'train' / 'labels').glob('*.txt'))
sample_labels = random.sample(label_files, k=min(5, len(label_files)))

class_counts = {0: 0, 1: 0}
bad_lines = []
for label_file in label_files:
    for line_no, line in enumerate(label_file.read_text().splitlines(), start=1):
        parts = line.split()
        if len(parts) != 5:
            bad_lines.append((label_file.name, line_no, line))
            continue
        cls = int(float(parts[0]))
        class_counts[cls] = class_counts.get(cls, 0) + 1

print('Train annotation counts:', class_counts)
print('Bad label lines:', len(bad_lines))
print('Sample label files:', [p.name for p in sample_labels])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

class_names = ['Fire', 'Smoke']
image_files = list((DATASET_DIR / 'train' / 'images').glob('*'))
sample_images = random.sample(image_files, k=min(4, len(image_files)))

fig, axes = plt.subplots(1, len(sample_images), figsize=(18, 5))
if len(sample_images) == 1:
    axes = [axes]

for ax, image_path in zip(axes, sample_images):
    img = Image.open(image_path).convert('RGB')
    w, h = img.size
    ax.imshow(img)
    label_path = DATASET_DIR / 'train' / 'labels' / f'{image_path.stem}.txt'
    if label_path.exists():
        for line in label_path.read_text().splitlines():
            cls, x, y, bw, bh = map(float, line.split())
            x1 = (x - bw / 2) * w
            y1 = (y - bh / 2) * h
            rect = patches.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2)
            ax.add_patch(rect)
            ax.text(x1, y1, class_names[int(cls)], color='white', backgroundcolor='black')
    ax.axis('off')
    ax.set_title(image_path.name[:30])

plt.tight_layout()